In [ ]:
import requests # à installer
from bs4 import BeautifulSoup # à installer
import pandas as pd # à installer
import time
import os
import re

In [ ]:
#Liste des fonctions utilisées

#Fonction visant à faire la recherche dans la BNF et à parser le résultat pour le rendre exploitable, ou à stopper le processus en cas de blocage
def requeteparsing(adresse):
    #print(adresse)
    try :
        response=requests.get(adresse)
        soup=BeautifulSoup(response.text, 'html.parser') 
        time.sleep(2) # 2 secondes de pause pour ne pas surcharger les services de la BNF
        if response.status_code != 200:
            raise RuntimeError("Trop de requête à la BNF. Attendre un peu et réessayer plus tard.")
    except:
        raise RuntimeError("Trop de requête à la BNF. Attendre un peu et réessayer plus tard.")
    return soup

#Fonction visant à récupérer les auteurs d'après la page BNF d'une oeuvre
def trouveauteurdepuisoeuvre(listurl):
    if listurl: #S'il y a des ouvrages proposés, prendre le premier
        url=listurl[0]
    else: #Sinon, on passe
        #print(listurl, "pass")
        return pd.Series({'Auteur_prédit':"",'Autre_auteur_prédit':""})
    #print(url)
    soup=requeteparsing(url)
    auteurs=[]
    authors=soup.find("p", id="auteur")
    if authors:
        ass=authors.find_all("a")
        for a in ass:
            if a.get("class")!=["pictos"]:
                auteurs.append(f"{a.text} [ https://catalogue.bnf.fr{a.get('href')} ]") #Le rôle de l'auteur s'il y en a un devrait être récupérable avec un a.next_sibling.strip()
    auteurs=";".join(auteurs)
    
    autre_auteurs=[]
    authors=soup.find("p", id="autreAuteur")
    if authors:
        ass=authors.find_all("a")
        for a in ass:
            if a.get("class")!=["pictos"]:
                autre_auteurs.append(f"{a.text} [ https://catalogue.bnf.fr{a.get('href')} ]") #Le rôle du para-auteur s'il y en a un devrait être récupérable avec un a.next_sibling.strip()
    autre_auteurs=";".join(autre_auteurs)

    return pd.Series({'Auteur_prédit':auteurs,'Autre_auteur_prédit':autre_auteurs})

#Petite fontion de nettoyage de chaîne de texte
def traitement(string):
    string=string.replace(' ','+')
    string=string.replace("(","").replace(")","")
    return string

#Fonction à faire la recherche automatique à la BNF de l'ouvrage
def trouveoeuvre(row):
    titre,auteur,date=row["Titre"],row["Auteur"],row["Année"]
    imprimeur=row["Impression"] if "Impression" in row.index else row["Imprimeur"] #La colonne n'a pas le même nom selon les csv ???!
    librairie1=row["Librairie 1"] if "Librairie 1" in row.index else row["Libraire 1"] #La colonne n'a pas le même nom selon les csv ???!
    #print("----------------------",auteur,"---",titre)
    if date:
        date=int(date)
        datef=f"{date-1}+{date}+{date+1}" #Ajouter date avant et après en possibilité pour augmenter chance de succès
    else:
        datef="1818+1819+1820+1821+1822+1823+1824+1825+1826+1827+1829+1830+1831+1832+1833+1834+1835+1836" #Si pas de date indiquée, donner toutes les dates possibles

    imprimeurf=[] #Récupération des noms de l'imprimeur et du libraire (parfois indiqué comme imprimeur dans BNF) pour la recherche
    imprimeur=imprimeur.strip()
    librairie1=librairie1.strip()
    if imprimeur:
        imprimeurf.append(re.search(r"(\b[^ ]+\b(?:\s+(?:fils|aîné|l'aîné|jeune))*)[.!?]?$", imprimeur).group(1)) #Ne garder que le dernier mot de l'imprimeur (souvent le nom) car le prénom est généralement abbrégé, afin de rendre ce nom obligatoire
    if librairie1:
        imprimeurf.append(re.search(r"(\b[^ ]+\b(?:\s+(?:fils|aîné|l'aîné|jeune))*)[.!?]?$", librairie1).group(1)) #Ne garder que le dernier mot de l'imprimeur (souvent le nom) car le prénom est généralement abbrégé, afin de rendre ce nom obligatoire
    imprimeurf="+".join(imprimeurf)

    if auteur: #Ne garder que le dernier mot avant virgule de chaque auteur (là où est presque toujours le nom de famille, élément le plus fiable pour recherche)
        listauteur=[]
        for auteurr in auteur.split(";"):
            auteurr=auteurr.split(",")[0]
            auteurr=auteurr.split(" ")[-1]
            listauteur.append(auteurr)
        auteurf="+".join(listauteur)
    else:
        auteurf=""

    #Récupérer un bon titre en supprimant des surtitres fréquents non repris dans BNF et en plaçant les sous-titre dans une recherche différente de celle du titre
    titre=titre if not titre.startswith("Répertoire du Théâtre-Français.") else titre[31:].strip()
    titre=titre if not titre.startswith("Bibliothèque française.") else titre[23:].strip()
    titre=titre if not titre.startswith("Classiques français.") else titre[20:].strip()
    titref=titre
    noticef=""
    if "," in titref:
        noticef=titref.split(",",1)[1]
        titref=titref.split(",",1)[0]
    if "." in titref:
        noticef+="+"+titref.split(".",1)[1]
        titref=titref.split(".",1)[0]
    if ";" in titref:
        noticef+="+"+titref.split(";",1)[1]
        titref=titref.split(";",1)[0]
    elif noticef=="" and titref.count(" ")>1:
        noticef=titref.split(" ",2)[2]
        titref=" ".join(titref.split(" ",2)[0:2])

    titref=traitement(titref)
    auteurf=traitement(auteurf)
    noticef=traitement(noticef)

    #Recherche 1 : le titre entier, une des date, les auteurs entiers, l'imprimeur/libraire et un des mots du sous-titre dans toute la notice
    adresse=f"https://catalogue.bnf.fr/search.do?mots0=TIT;-1;0;{titref}&mots1=DAT;0;1;{datef}&mots2=NRI;0;0;{auteurf}&mots3=NRC;0;1;{imprimeurf.replace(' ','+')}&mots4=ALL;0;1;{noticef.replace(' ','+')}&facDocs=a&&pageRech=rav"
    soup=requeteparsing(adresse)
    listerecherche=soup.find("ul", class_="liste-notices")
    if listerecherche:
        lis=listerecherche.find_all('li')
        #print(len(lis))
        resultat=['https://catalogue.bnf.fr'+li.find("a").get('href') for li in lis]
        #print(resultat)
    else:
        #Recherche 2 : le titre entier, une des date, les auteurs entiers (plus l'imprimeur/libraire) et un des mots du sous-titre dans toute la notice
        adresse=f"https://catalogue.bnf.fr/search.do?mots0=TIT;-1;0;{titref.replace(' ','+')}&mots1=DAT;0;1;{datef}&mots2=NRI;0;0;{auteurf}&mots3=ALL;0;1;{noticef.replace(' ','+')}&facDocs=a&&pageRech=rav"
        soup=requeteparsing(adresse)
        listerecherche=soup.find("ul", class_="liste-notices")
        if listerecherche:
            lis=listerecherche.find_all('li')
            #print(len(lis))
            resultat=['https://catalogue.bnf.fr'+li.find("a").get('href') for li in lis]
            #print(resultat)
        else:
            #print("Pas d'ouvrage trouvé.")
            resultat=[]
    #print(row['Identifiant ouvrage']) #Par comparaison
    return resultat,pd.Series({'Dernier lien de recherche':adresse,'Ouvrage_prédit':" , ".join(resultat)})

#Fonction visant à appliquer cette recherche sur toutes les lignes d'un tableau une à une et à les sauvegarder au fur-et-à-mesure sur un autre
def traitementcsv(adresse_fichiercsv_a_traiter,adresse_fichiercsv_deja_traite):
    df=pd.read_csv(adresse_fichiercsv_a_traiter, encoding="utf-8",sep=";",header=0,dtype=str)
    df = df.fillna("")
    if os.path.exists(adresse_fichiercsv_deja_traite):
        dfdeja=pd.read_csv(adresse_fichiercsv_deja_traite, encoding="utf-8",sep=";",header=0,dtype=str).drop(columns=["Dernier lien de recherche","Ouvrage_prédit","Auteur_prédit","Autre_auteur_prédit"])
        dfdeja = dfdeja.fillna("")
        nb_lines_deja_traitees=set(map(tuple, dfdeja.to_numpy()))
    else:
        colonnes=list(df.columns)+["Dernier lien de recherche","Ouvrage_prédit","Auteur_prédit","Autre_auteur_prédit"]
        pd.DataFrame(columns=colonnes).to_csv(adresse_fichiercsv_deja_traite,encoding="utf-8",sep=";",index=False)
        nb_lines_deja_traitees=set()
    for _,row in df.iterrows():
        if tuple(row.values) in nb_lines_deja_traitees: #Ne pas traiter les lignes déjà faites (Ne retraitera pas à non plus les doublons du cv s'il y en a)
            continue
        ouvrages_possibles_list,ouvrages_possibles_serie=trouveoeuvre(row)
        auteurs_trouves=trouveauteurdepuisoeuvre(ouvrages_possibles_list)
        nouvelle_ligne=pd.concat([row,ouvrages_possibles_serie,auteurs_trouves])
        pd.DataFrame([nouvelle_ligne]).to_csv(adresse_fichiercsv_deja_traite,mode="a",encoding="utf-8",sep=";",header=False,index=False)
    os.rename(adresse_fichiercsv_deja_traite, adresse_fichiercsv_deja_traite.replace("@EN_TRAITEMENT_",""))
    #print(f"         Traité : {adresse_fichiercsv_a_traiter}")
    return
def traitement_touscsvs(lien_csvs):
    dossier_traitement=os.path.join(lien_csvs,"données_traitées")
    os.makedirs(dossier_traitement, exist_ok=True)
    liste_deja_traitee=[csv for csv in os.listdir(dossier_traitement) if csv.endswith(".csv")]
    liste_a_traiter=[csv for csv in os.listdir(lien_csvs) if csv.endswith(".csv") and csv not in liste_deja_traitee]
    for csv in liste_a_traiter:
        csv_base=os.path.join(lien_csvs,csv)
        csv_en_traitement=os.path.join(dossier_traitement,"@EN_TRAITEMENT_"+csv)
        traitementcsv(csv_base,csv_en_traitement)

In [ ]:
adresse=r"C:\Users\xxx\data" #Adresse du dossier contenant les csv à récupérer sur votre PC
traitement_touscsvs(adresse)